In [1]:
import keras
import tensorflow as tf
import numpy as np
from sklearn.metrics import confusion_matrix
import pandas as pd


In [4]:
base_dir=r"D:\training\skenario\pertama"
img_size=(244, 244)
batch_size= 32

In [5]:
train_ds= tf.keras.utils.image_dataset_from_directory(
    base_dir +"/train",
    image_size= img_size,
    batch_size= batch_size,
    label_mode= "categorical"
)

Found 2800 files belonging to 4 classes.


In [6]:
val_ds= tf.keras.utils.image_dataset_from_directory(
    base_dir +"/val",
    image_size= img_size,
    batch_size= batch_size,
    label_mode= "categorical"
)

Found 600 files belonging to 4 classes.


In [7]:
test_ds= tf.keras.utils.image_dataset_from_directory(
    base_dir +"/test",
    class_names= ['Conversion', 'Real', 'Record', 'Syntesys'],
    label_mode='int',
    image_size= img_size,
    batch_size= batch_size,
    shuffle= False
)

ValueError: The `class_names` passed did not match the names of the subdirectories of the target directory. Expected: ['conversion', 'real', 'record', 'syntesis'] (or a subset of it), but received: class_names=['Conversion', 'Real', 'Record', 'Syntesys']

In [ ]:
class MyModel(keras.Model):
    def __init__(self):
        super().__init__()
        self.rescall= keras.layers.Rescaling(1./255)
        self.conv2d1= keras.layers.Conv2D(
            filters=32, kernel_size=(3, 3), activation='relu'
        )
        self.pool1= keras.layers.MaxPooling2D()

        self.conv2d2= keras.layers.Conv2D(
            filters=64, kernel_size=(3, 3), activation='relu'
        )
        self.pool2= keras.layers.MaxPooling2D()
        self.conv2d3= keras.layers.Conv2D(
            filters=128, kernel_size=(3, 3), activation='relu'
         )
        self.pool3= keras.layers.MaxPooling2D()
        self.flatten= keras.layers.Flatten()
        self.dense1= keras.layers.Dense(128, activation= 'relu')
        self.dense2= keras.layers.Dense(4, activation= 'softmax' )
    
    def call(self,x):
        x= self.rescall(x)
        x= self.pool1(self.conv2d1(x))
        x= self.pool2(self.conv2d2(x))
        x= self.pool3(self.conv2d3(x))
        x= self.flatten(x)
        x= self.dense1(x)
        return self.dense2(x)
        



In [8]:
model= MyModel()

opt= keras.optimizers.Adam(learning_rate=0.01)

model.compile(
    optimizer= opt,
    loss='sparse_categorical_crossentropy',
    metrics=["accuracy"]
)

In [9]:
history= model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

Epoch 1/15


ValueError: Argument `output` must have rank (ndim) `target.ndim - 1`. Received: target.shape=(None, 4), output.shape=(None, 4)

In [ ]:
y_true_oh= np.concatenate([y.numpy() for x, y in test_ds], axis=0 )
y_true= np.argmax(y_true_oh, axis=1)
y_pred= np.argmax(model.predict(test_ds), axis=1)

label= ['Conversion', 'Real', 'Record', 'Syntesys']
cm= confusion_matrix(y_true, y_pred)
cm_df= pd.DataFrame(cm, index=label, columns=label)

print(cm_df)

In [26]:
y_true= np.concatenate([y.numpy() for x, y in test_ds], axis=0 )
y_pred= np.argmax(model.predict(test_ds), axis=1)

label= ['Conversion', 'Real', 'Record', 'Syntesys']
cm= confusion_matrix(y_true, y_pred)
cm_df= pd.DataFrame(cm, index=label, columns=label)

print(cm_df)


19/19 ━━━━━━━━━━━━━━━━━━━━ 11s 532ms/step
            Conversion  Real  Record  Syntesys
Conversion           0     3       0       147
Real                 0     1       0       149
Record               0     0       0       150
Syntesys             0     7       0       143


In [18]:
print(train_ds.class_names)
print(test_ds.class_names)

['Conversion', 'Real', 'Record', 'Syntesys']
['Conversion', 'Real', 'Record', 'Syntesys']
